# 01 — Chunking

Fase pertama indexing: memotong dokumen besar menjadi potongan kecil (chunks)
yang siap di-embed oleh model `thenlper/gte-large`.

**Spec model yang mempengaruhi chunking:**
- Max sequence: 512 token (~2048 karakter)
- Safe `chunk_size`: 600–800 karakter
- `chunk_overlap`: 80–120 karakter

## 0. Cek Library
Pastikan semua library yang dibutuhkan sudah terinstall.

In [1]:
import importlib

required = ["langchain", "langchain_community", "pypdf"]

for lib in required:
    found = importlib.util.find_spec(lib.replace("-", "_")) is not None
    status = "✓ OK" if found else "✗ MISSING — pip install " + lib
    print(f"{lib:25s} {status}")

langchain                 ✓ OK
langchain_community       ✓ OK
pypdf                     ✓ OK


## 1. Load Dokumen
Baca PDF halaman per halaman menggunakan PyPDFLoader.
Setiap halaman jadi satu objek `Document` dengan `page_content` dan `metadata`.

In [3]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "../data/PHIS_SDS_SAM_1.1.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Total halaman  : {len(pages)}")
print(f"Tipe objek     : {type(pages[0])}")
print(f"\n--- Metadata halaman pertama ---")
print(pages[0].metadata)
print(f"\n--- Cuplikan teks (500 karakter pertama) ---")
print(pages[0].page_content[:500])

Total halaman  : 271
Tipe objek     : <class 'langchain_core.documents.base.Document'>

--- Metadata halaman pertama ---
{'producer': '', 'creator': 'WPS Writer', 'creationdate': '2026-05-28T10:52:36+08:00', 'author': 'dataware05', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2026-05-28T10:52:36+08:00', 'sourcemodified': "D:20260528105236+08'00'", 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/PHIS_SDS_SAM_1.1.pdf', 'total_pages': 271, 'page': 0, 'page_label': '1'}

--- Cuplikan teks (500 karakter pertama) ---
SYSTEM DESIGN
SPECIFICATION (SDS)
Pharmacy Information System (PhIS)
Special Approval Medicine (SAM)
DOCUMENT DATE : 06/05/2025
DOCUMENT VERSION : 1.0
DOCUMENT ID : PhIS/SDS/SAM


## 2. Kenapa Perlu Chunking?
Model `gte-large` hanya bisa proses 512 token (~2048 karakter) sekali jalan.
Kita cek dulu berapa halaman yang melebihi batas aman

In [4]:
lengths = [len(p.page_content) for p in pages]

print(f"Karakter per halaman:")
print(f"  Min  : {min(lengths)}")
print(f"  Max  : {max(lengths)}")
print(f"  Rata : {sum(lengths) // len(lengths)}")
print(f"\nHalaman yang melebihi batas aman gte-large (1600 karakter):")
over = [(i, l) for i, l in enumerate(lengths) if l > 1600]
print(f"  {len(over)} dari {len(pages)} halaman perlu di-chunk")

Karakter per halaman:
  Min  : 60
  Max  : 5159
  Rata : 1241

Halaman yang melebihi batas aman gte-large (1600 karakter):
  100 dari 271 halaman perlu di-chunk


## 3. Chunking dengan RecursiveCharacterTextSplitter
Splitter terbaik untuk dokumen umum. Memotong di `\n\n` dulu,
lalu `\n`, lalu `.`, lalu spasi — menghormati struktur paragraf.

- `chunk_size = 700` → aman untuk gte-large (512 token max)
- `chunk_overlap = 100` → menjaga konteks di batas antar chunk

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = splitter.split_documents(pages)

chunk_lengths = [len(c.page_content) for c in chunks]

print(f"Total chunks   : {len(chunks)}")
print(f"Min panjang    : {min(chunk_lengths)} karakter")
print(f"Max panjang    : {max(chunk_lengths)} karakter")
print(f"Rata-rata      : {sum(chunk_lengths) // len(chunk_lengths)} karakter")
print(f"\nChunk di atas 1600 karakter (bahaya untuk gte-large):")
over = [l for l in chunk_lengths if l > 1600]
print(f"  {len(over)} chunk")

Total chunks   : 666
Min panjang    : 60 karakter
Max panjang    : 700 karakter
Rata-rata      : 548 karakter

Chunk di atas 1600 karakter (bahaya untuk gte-large):
  0 chunk


## 4. Inspect Hasil Chunk
Lihat isi chunk satu per satu untuk memastikan hasilnya masuk akal.

In [7]:
def inspect_chunk(chunks, index):
    c = chunks[index]
    bar = "=" * 55
    print(bar)
    print(f"Index       : {index}")
    print(f"Halaman PDF : {c.metadata.get('page', 'N/A')}")
    print(f"Panjang     : {len(c.page_content)} karakter")
    print(bar)
    print(c.page_content)
    print(bar)

# Ganti angka untuk lihat chunk berbeda
inspect_chunk(chunks, 0)
print()
inspect_chunk(chunks, 1)

Index       : 0
Halaman PDF : 0
Panjang     : 177 karakter
SYSTEM DESIGN
SPECIFICATION (SDS)
Pharmacy Information System (PhIS)
Special Approval Medicine (SAM)
DOCUMENT DATE : 06/05/2025
DOCUMENT VERSION : 1.0
DOCUMENT ID : PhIS/SDS/SAM

Index       : 1
Halaman PDF : 1
Panjang     : 665 karakter
System Design Specification (SDS)
Ref: PhIS/SDS/SAM Page i
DOCUMENT INFORMATION
This document details the system design for the Pharmacy Information System (PhIS), offering
a structured framework that defines the system's functionality and user interactions. It highlights
key components of the design, including:
1. User Interface Design: This section explains the structure of the user interface (UI) to
deliver a seamless experience for end-users. It details the visual and functional
elements, including navigation, input forms, data displays, and interactive features. The
design emphasizes user-friendliness, accessibility, and efficiency, ensuring the system


## 5. Lihat Efek chunk_overlap
Buktikan bahwa teks di batas antar chunk saling overlap.

In [10]:
def show_boundary(chunks, i):
    print(f"--- Akhir chunk {i} ---")
    print(repr(chunks[i].page_content[-120:]))
    print(f"\n--- Awal chunk {i+1} ---")
    print(repr(chunks[i+1].page_content[:120]))
    print()
print(f"Total chunks: {len(chunks)}")
show_boundary(chunks, 1)
show_boundary(chunks, 2)

# Hitung berapa chunk per halaman
from collections import Counter

pages_count = Counter(c.metadata['page'] for c in chunks)
print(f"Halaman dengan 1 chunk  : {sum(1 for v in pages_count.values() if v == 1)}")
print(f"Halaman dengan 2 chunks : {sum(1 for v in pages_count.values() if v == 2)}")
print(f"Halaman dengan 3 chunks : {sum(1 for v in pages_count.values() if v == 3)}")
print(f"Halaman dengan 4+ chunks: {sum(1 for v in pages_count.values() if v >= 4)}")

Total chunks: 666
--- Akhir chunk 1 ---
's, and interactive features. The\ndesign emphasizes user-friendliness, accessibility, and efficiency, ensuring the system'

--- Awal chunk 2 ---
'design emphasizes user-friendliness, accessibility, and efficiency, ensuring the system\naccommodates the varied needs of'

--- Akhir chunk 2 ---
' serves as a critical reference\nfor the development team, providing technical specifications and implementation details.'

--- Awal chunk 3 ---
'for the development team, providing technical specifications and implementation details.\nIt defines system requirements,'

Halaman dengan 1 chunk  : 86
Halaman dengan 2 chunks : 66
Halaman dengan 3 chunks : 46
Halaman dengan 4+ chunks: 73


## 7. Simpan Chunks ke File
Simpan hasil chunking ke JSON supaya bisa dipakai di notebook berikutnya
(02_embedding.ipynb) tanpa perlu load ulang PDF.

In [11]:
import json
import os

data = [
    {
        "index": i,
        "page_content": c.page_content,
        "metadata": c.metadata
    }
    for i, c in enumerate(chunks)
]

os.makedirs("../data", exist_ok=True)

with open("../data/chunks.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Tersimpan : ../data/chunks.json")
print(f"Total     : {len(data)} chunks")
print(f"\nSiap lanjut ke 02_embedding.ipynb!")

Tersimpan : ../data/chunks.json
Total     : 666 chunks

Siap lanjut ke 02_embedding.ipynb!
